In [1]:
import numpy as np
import re
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Embedding, Dropout, LSTM, Bidirectional
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from datasets import load_dataset


In [2]:
dataset = load_dataset("agentlans/project-gutenberg-top100", "en")
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['file', 'index', 'text'],
        num_rows: 224639
    })
})


In [3]:
shuffled = dataset['train'].shuffle(seed=42)
raw_paragraphs = shuffled['text']

clean_paragraphs = [p.strip() for p in raw_paragraphs if p and p.strip()]

MAX_LINES = 4000
clean_paragraphs = clean_paragraphs[:MAX_LINES]

doc = '\n'.join(clean_paragraphs)

print(f"Paragraphs used:   {len(clean_paragraphs):,} / {len(raw_paragraphs):,}")
print(f"Total characters:  {len(doc):,}")
print(f"Approx word count: {len(doc.split()):,}")


Paragraphs used:   4,000 / 224,639
Total characters:  1,244,064
Approx word count: 224,152


In [4]:
def get_sentences(text):
    """Split corpus into individual sentences for proper context learning."""
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.split()) >= 3]
    return sentences

sentences = get_sentences(doc)
print(f"Total sentences extracted: {len(sentences)}")
print(f"\nExample sentences:")
for s in sentences[:3]:
    print(f"  → {s}")


Total sentences extracted: 9550

Example sentences:
  → What had he to do, indeed?
  → He did not care for cards; he did not go to a club.
  → Spending the time with jovial gentlemen of Oblonsky's type—she knew now what that meant ...


In [5]:
nw = 10000
tokenizer = Tokenizer(num_words = nw, oov_token='<OOV>')
tokenizer.fit_on_texts(sentences)

vocab_size = len(tokenizer.word_index) + 1
print(f"Vocabulary size: {vocab_size} unique words")

input_sequences = []
for sentence in sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

print(f"Total training sequences: {len(input_sequences)}")
print(f"Shortest sequence length: {min(len(s) for s in input_sequences)}")
print(f"Longest sequence length:  {max(len(s) for s in input_sequences)}")


Vocabulary size: 19689 unique words
Total training sequences: 215317
Shortest sequence length: 2
Longest sequence length:  315


In [6]:
context_window = 10
max_sequence_len = context_window
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre', truncating='pre')
)

x = input_sequences[:, :-1]
y = input_sequences[:, -1]

print(f"x shape: {x.shape}   ← input to model (sequences)")
print(f"y shape: {y.shape}   ← integer labels (next-word index)")
print(f"input_length for Embedding: {x.shape[1]}")


x shape: (215317, 9)   ← input to model (sequences)
y shape: (215317,)   ← integer labels (next-word index)
input_length for Embedding: 9


In [7]:
input_len = x.shape[1]

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=input_len))
model.add(LSTM(128))
model.add(Dropout(0.2))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


c:\Users\sahoo\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
early_stop = EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    x, y,
    epochs=100,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\nTraining stopped at epoch: {len(history.history['loss'])}")
print(f"Best loss: {min(history.history['loss']):.4f}")


Epoch 1/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 78s 45ms/step - accuracy: 0.0788 - loss: 6.5692 - learning_rate: 0.0010
Epoch 2/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 58s 34ms/step - accuracy: 0.1140 - loss: 6.0286 - learning_rate: 0.0010
Epoch 3/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 68s 40ms/step - accuracy: 0.1302 - loss: 5.7528 - learning_rate: 0.0010
Epoch 4/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 58s 34ms/step - accuracy: 0.1397 - loss: 5.5612 - learning_rate: 0.0010
Epoch 5/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 61s 36ms/step - accuracy: 0.1473 - loss: 5.3994 - learning_rate: 0.0010
Epoch 6/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 59s 35ms/step - accuracy: 0.1539 - loss: 5.2464 - learning_rate: 0.0010
Epoch 7/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 60s 36ms/step - accuracy: 0.1590 - loss: 5.1048 - learning_rate: 0.0010
Epoch 8/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 61s 36ms/step - accuracy: 0.1648 - loss: 4.9653 - learning_rate: 0.0010
Epoch 9/100
1683/1683 ━━━━━━━━━━━━━━━━━━━━ 63s 37ms/step - accuracy: 0.1697 - lo

In [ ]:
model.save('next_word_model.keras')

with open('tokenizer.pkl', 'wb') as f:  
    pickle.dump(tokenizer, f)

with open('model_config.pkl', 'wb') as f:
    pickle.dump({'max_sequence_len': max_sequence_len, 'vocab_size': vocab_size}, f)

print("Saved: next_word_model.keras")
print("Saved: tokenizer.pkl")
print("Saved: model_config.pkl")


Saved: next_word_model.keras
Saved: tokenizer.pkl
Saved: model_config.pkl


In [10]:
def predict_next_words(model, tokenizer, text, max_sequence_len, top_k=5):
    seq = tokenizer.texts_to_sequences([text.lower()])[0]
    if not seq:
        return [("(no known words in input)", 0.0)]

    seq = pad_sequences([seq], maxlen=max_sequence_len - 1, padding='pre')

    pred = model.predict(seq, verbose=0)[0]

    top_indices = np.argsort(pred)[::-1][:top_k]

    results = []
    for idx in top_indices:
        word = tokenizer.index_word.get(idx, '<unknown>')
        confidence = float(pred[idx])
        results.append((word, confidence))

    return results


def predict_with_temperature(model, tokenizer, text, max_sequence_len, temperature=1.0):
    seq = tokenizer.texts_to_sequences([text.lower()])[0]
    if not seq:
        return "(no known words)", 0.0

    seq = pad_sequences([seq], maxlen=max_sequence_len - 1, padding='pre')
    pred = model.predict(seq, verbose=0)[0].astype('float64')

    pred = np.log(pred + 1e-10) / temperature
    pred = np.exp(pred)
    pred = pred / pred.sum()

    sampled_index = np.random.choice(len(pred), p=pred)
    word = tokenizer.index_word.get(sampled_index, '<unknown>')
    confidence = float(pred[sampled_index])

    return word, confidence


In [11]:
test_inputs = [
    "the sun was",
    "deep in the heart of the",
    "at night the sky",
    "the ancient castle",
    "people were out enjoying",
]

print("=" * 60)
print("\tTOP-K SUGGESTIONS (greedy, deterministic)")
print("=" * 60)
for text in test_inputs:
    suggestions = predict_next_words(model, tokenizer, text, max_sequence_len, top_k=5)
    print(f"\nInput: '{text}'")
    for rank, (word, conf) in enumerate(suggestions, 1):
        bar = '█' * int(conf * 30)
        print(f"  {rank}. {word:<20} {conf:.4f}  {bar}")

print("\n" + "=" * 60)
print("\tTEMPERATURE SAMPLING (creative mode)")
print("=" * 60)
for T in [0.5, 1.0, 1.5]:
    text = "the forest was"
    word, conf = predict_with_temperature(model, tokenizer, text, max_sequence_len, temperature=T)
    print(f"T={T}  → '{word}'  (confidence: {conf:.4f})")


	TOP-K SUGGESTIONS (greedy, deterministic)

Input: 'the sun was'
  1. <OOV>                0.2727  ████████
  2. all                  0.1257  ███
  3. still                0.0582  █
  4. very                 0.0404  █
  5. like                 0.0343  █

Input: 'deep in the heart of the'
  1. castle               0.1169  ███
  2. narrow               0.1149  ███
  3. <OOV>                0.1058  ███
  4. room                 0.0655  █
  5. head                 0.0250  

Input: 'at night the sky'
  1. was                  0.5982  █████████████████
  2. followed             0.1277  ███
  3. lord                 0.0311  
  4. arose                0.0215  
  5. god                  0.0161  

Input: 'the ancient castle'
  1. of                   0.3090  █████████
  2. which                0.1920  █████
  3. complete             0.1626  ████
  4. between              0.0683  ██
  5. on                   0.0500  █

Input: 'people were out enjoying'
  1. these                0.3416  ██████████

In [12]:
def generate_text(seed_text, model, tokenizer, max_sequence_len,
                  num_words=10, temperature=0.8):
    """
    Generate `num_words` words after seed_text using temperature sampling.
    Each predicted word becomes part of the next input → context carries forward.
    """
    output = seed_text
    for _ in range(num_words):
        next_word, _ = predict_with_temperature(
            model, tokenizer, output, max_sequence_len, temperature=temperature
        )
        if next_word in ('<unknown>', '<OOV>'):
            break
        output += ' ' + next_word
    return output


seeds = [
    "the castle stood on",
    "deep in the amazon",
    "at night the stars",
]

print("Generated sentences (temperature=0.8):")
print("-" * 50)
for seed in seeds:
    result = generate_text(seed, model, tokenizer, max_sequence_len, num_words=8)
    print(f"Seed:   '{seed}'")
    print(f"Output: '{result}'")
    print()


Generated sentences (temperature=0.8):
--------------------------------------------------
Seed:   'the castle stood on'
Output: 'the castle stood on his'

Seed:   'deep in the amazon'
Output: 'deep in the amazon opposite the room sent some woman farther for'

Seed:   'at night the stars'
Output: 'at night the stars he also'

